# **Dataset**

In [21]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/pythonafroz/medquad-medical-question-answer-for-ai-research/medquad.csv")

df.head()

,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


In [22]:
print(df.columns)
print(len(df))

Index(['question', 'answer', 'source', 'focus_area'], dtype='object')
16412


In [23]:
df = df[['question', 'answer']].dropna()
df = df.drop_duplicates()

print(len(df))

16359


In [24]:
df.tail()

,question,answer
16407,What is (are) Diabetic Neuropathies: The Nerve...,Focal neuropathy appears suddenly and affects ...
16408,How to prevent Diabetic Neuropathies: The Nerv...,The best way to prevent neuropathy is to keep ...
16409,How to diagnose Diabetic Neuropathies: The Ner...,Doctors diagnose neuropathy on the basis of sy...
16410,What are the treatments for Diabetic Neuropath...,The first treatment step is to bring blood glu...
16411,What to do for Diabetic Neuropathies: The Nerv...,- Diabetic neuropathies are nerve disorders ca...


In [25]:
df = df.sample(2000, random_state=42)
df = df.reset_index(drop=True)

In [26]:
len(df)

2000

In [27]:
df.to_csv("dataset")

In [28]:
def format_example(row):
    question = str(row["question"]).strip()
    answer = str(row["answer"]).strip()

    return f"### Instruction:\n{question}\n\n### Response:\n{answer}"

In [29]:
df["text"] = df.apply(format_example, axis=1)

In [30]:
df["text"].to_csv("formatted_dataset", index=False)

In [31]:
print(df["text"][2])

### Instruction:
What are the symptoms of Idiopathic inflammatory myopathy ?

### Response:
What are the signs and symptoms of Idiopathic inflammatory myopathy? The Human Phenotype Ontology provides the following list of signs and symptoms for Idiopathic inflammatory myopathy. If the information is available, the table below includes how often the symptom is seen in people with this condition. You can use the MedlinePlus Medical Dictionary to look up the definitions for these medical terms. Signs and Symptoms Approximate number of patients (when available) Autosomal dominant inheritance - Myositis - Proximal muscle weakness - The Human Phenotype Ontology (HPO) has collected information on how often a sign or symptom occurs in a condition. Much of this information comes from Orphanet, a European rare disease database. The frequency of a sign or symptom is usually listed as a rough estimate of the percentage of patients who have that feature. The frequency may also be listed as a fractio

In [33]:
from datasets import Dataset

dataset = Dataset.from_pandas(df[["text"]])

In [34]:
print(dataset[0])

{'text': '### Instruction:\nIs Galactosialidosis inherited ?\n\n### Response:\nHow is galactosialidosis inherited? Galactosialidosis is inherited in an autosomal recessive pattern, which means both copies of the gene in each cell have mutations. The parents of an individual with an autosomal recessive condition each carry one copy of the mutated gene, but they typically do not show signs and symptoms of the condition.'}


# **TRAINING** 

In [35]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 37.4 MB/s eta 0:00:00


In [37]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [38]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [42]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="epoch",
    max_length=512,
    fp16=False,
    bf16=False,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Step,Training Loss
10,1.745709
20,1.674860
30,1.525306
40,1.424060
50,1.405077
60,1.342108
70,1.296485
80,1.268967
90,1.250988
100,1.168367


TrainOutput(global_step=250, training_loss=1.2783007774353028, metrics={'train_runtime': 571.3031, 'train_samples_per_second': 3.501, 'train_steps_per_second': 0.438, 'total_flos': 4633617764376576.0, 'train_loss': 1.2783007774353028})

In [43]:
model.save_pretrained("tinyllama-medquad-qlora")
tokenizer.save_pretrained("tinyllama-medquad-qlora")

('tinyllama-medquad-qlora/tokenizer_config.json',
 'tinyllama-medquad-qlora/chat_template.jinja',
 'tinyllama-medquad-qlora/tokenizer.json')

In [44]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

model = PeftModel.from_pretrained(
    base_model,
    "tinyllama-medquad-qlora"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [48]:
prompt = """### Instruction:
Who is at risk for Coronary Heart Disease Risk Factors?

### Response:
"""

In [49]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.7,
    top_p=0.9
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

### Instruction:
Who is at risk for Coronary Heart Disease Risk Factors?

### Response:
The risk factors for coronary heart disease are the same as those for heart disease in general. These include high blood pressure, high cholesterol, smoking, and a sedentary lifestyle.


In [ ]:
from huggingface_hub import login

login("")

In [55]:
model.push_to_hub("Sanjarbek1024/tinyllama-medquad-qlora")
tokenizer.push_to_hub("Sanjarbek1024/tinyllama-medquad-qlora")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Sanjarbek1024/tinyllama-medquad-qlora/commit/0839949e9922a6ecd24d06d97c63c20696b73921', commit_message='Upload tokenizer', commit_description='', oid='0839949e9922a6ecd24d06d97c63c20696b73921', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Sanjarbek1024/tinyllama-medquad-qlora', endpoint='https://huggingface.co', repo_type='model', repo_id='Sanjarbek1024/tinyllama-medquad-qlora'), pr_revision=None, pr_num=None)

In [56]:
model = model.merge_and_unload()
model.save_pretrained("merged-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [58]:
model.push_to_hub("Sanjarbek1024/tinyllama-medquad-merged")
tokenizer.push_to_hub("Sanjarbek1024/tinyllama-medquad-merged")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Sanjarbek1024/tinyllama-medquad-merged/commit/c3a0f2746d93373715f9660875f824b1a640791e', commit_message='Upload tokenizer', commit_description='', oid='c3a0f2746d93373715f9660875f824b1a640791e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Sanjarbek1024/tinyllama-medquad-merged', endpoint='https://huggingface.co', repo_type='model', repo_id='Sanjarbek1024/tinyllama-medquad-merged'), pr_revision=None, pr_num=None)